# 02- Vector_Search Homework solutions:

In [4]:
%%bash
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
wget "$PREFIX/download.py"
wget "$PREFIX/embedder.py"

--2026-06-29 19:13:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

     0K .                                                     100% 34.1M=0s

2026-06-29 19:13:40 (34.1 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-29 19:13:40--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
L

Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31

-0.02

-0.12

-0.44

In [2]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q)
print(v1[0])

2026-06-29 21:38:22.220363238 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893


Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Each document is a dictionary with a filename and content, and there are 72 pages.



#### Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

 0.07

 0.37

 0.68
 
 0.92

In [4]:
import numpy as np

# 1. Grab the content from index 22
content = documents[22]['content']

# 2. Embed this specific page
v2 = embed.encode(content)

# 3. Compute the similarity with your Q1 query vector (v1)
similarity = np.dot(v1, v2)

print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.3611


### Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)
Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md
02-vector-search/lessons/06-rag-vector.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/09-onnx-embedder.md

v1.dot(dv)


In [5]:
import numpy as np
from gitsource import chunk_documents

In [6]:
# 1. Chunk your entire documents collection exactly as described
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 295


In [7]:
chunk_texts = [chunk['content'] for chunk in chunks]

In [8]:
X = embed.encode_batch(chunk_texts)

In [10]:
# 4. Compute similarities and locate the maximum score index
scores = X.dot(v1)
best_idx = np.argmax(scores)

In [11]:

print("Highest Similarity Score:", scores[best_idx])
print("Winning File Path Source:", chunks[best_idx]['filename'])

Highest Similarity Score: 0.6489016436447387
Winning File Path Source: 02-vector-search/lessons/07-sqlitesearch-vector.md


### Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

02-vector-search/lessons/04-vector-search.md

04-evaluation/lessons/05-search-metrics.md

04-evaluation/lessons/13-llm-as-judge.md

05-monitoring/lessons/04-metrics.md




In [20]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [24]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=1)

In [25]:
# 5. Extract and print the filename of the first result
winning_document = results[0]
print("Top Result Filename:", winning_document['filename'])

Top Result Filename: 04-evaluation/lessons/05-search-metrics.md


### Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

02-vector-search/lessons/01-intro.md

02-vector-search/lessons/02-embeddings.md

02-vector-search/lessons/08-pgvector.md

03-orchestration/lessons/05-rag.md

In [28]:
from minsearch import Index, VectorSearch

# 1. Initialize and build the traditional keyword index
text_index = Index(keyword_fields=["content"], text_fields=["content"])
text_index.fit(chunks)

# 2. Initialize and build the vector search index
vector_index = VectorSearch()
vector_index.fit(X, chunks)

# 3. Define the Q5 query text and embed it
query_text = "How do I store vectors in PostgreSQL?"
v5 = embed.encode(query_text)

# 4. Run both search methods for the top 5 results
text_results = text_index.search(query_text, num_results=5)
vector_results = vector_index.search(v5, num_results=5)

# 5. Extract filenames to compare
text_files = {doc['filename'] for doc in text_results}
vector_files = {doc['filename'] for doc in vector_results}

# Find what is unique to the vector search
unique_to_vector = vector_files - text_files
print("Unique to vector search results:", unique_to_vector)

Unique to vector search results: {'02-vector-search/lessons/08-pgvector.md'}


### Q6. Hybrid search
Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))
"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [37]:
# 1. Correct the vector search method call using query_vector=
query_text = "How do I give the model access to tools?"
v6 = embed.encode(query_text)

# Top 5 for text search
text_results = text_index.search(query_text, num_results=5)

# Top 5 for vector search (Fixing the AttributeError here)
vector_results = vector_index.search(query_vector=v6, num_results=5)

# 2. Define the RRF function provided in the question
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            # Using filename and text content slice to form a unique key
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 3. Fuse the results
fused_results = rrf([vector_results, text_results])

# 4. Print the top ranked document
print(" RRF First Place Winner:", fused_results[0]["filename"])

 RRF First Place Winner: 01-agentic-rag/lessons/13-function-calling.md
